In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam



In [5]:
df= pd.read_csv("IMDB Dataset.csv")

In [6]:
# Labels
df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})

# Raw text and labels
X_text = df["review"].astype(str).values
y = df["label"].values


In [7]:
# Split data
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)


In [8]:
# Tokenization
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq = tokenizer.texts_to_sequences(X_val_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

# Padding
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

In [10]:
# LSTM model
model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=max_words, output_dim=64),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

In [11]:
# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nModel summary:")
model.summary()



Model summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 673,089 (2.57 MB)

 Trainable params: 673,089 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:

# Train
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 75s 67ms/step - accuracy: 0.5371 - loss: 0.6840 - val_accuracy: 0.5847 - val_loss: 0.6600
Epoch 2/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 73s 67ms/step - accuracy: 0.7701 - loss: 0.4903 - val_accuracy: 0.8475 - val_loss: 0.3733
Epoch 3/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 75s 69ms/step - accuracy: 0.8833 - loss: 0.3005 - val_accuracy: 0.8627 - val_loss: 0.3185
Epoch 4/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 79s 72ms/step - accuracy: 0.9152 - loss: 0.2272 - val_accuracy: 0.8747 - val_loss: 0.3281
Epoch 5/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 76s 70ms/step - accuracy: 0.9421 - loss: 0.1681 - val_accuracy: 0.8605 - val_loss: 0.4118
Epoch 6/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 75s 68ms/step - accuracy: 0.9623 - loss: 0.1162 - val_accuracy: 0.8655 - val_loss: 0.4373
Epoch 7/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 76s 69ms/step - accuracy: 0.9759 - loss: 0.0804 - val_accuracy: 0.8569 - val_loss: 0.4728
Epoch 8/10
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 75s 68ms/step - accuracy: 0.9837 -

In [14]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print("\nTest Results")
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

# Predict
y_prob = model.predict(X_test_pad, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))


Test Results
Test Loss: 0.6963499188423157
Test Accuracy: 0.8507999777793884

Accuracy: 0.8508

Confusion Matrix:
[[3388  362]
 [ 757 2993]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.82      0.90      0.86      3750
    positive       0.89      0.80      0.84      3750

    accuracy                           0.85      7500
   macro avg       0.85      0.85      0.85      7500
weighted avg       0.85      0.85      0.85      7500

